In [46]:
import pandas as pd
import sys
import os
import numpy as np
import attr

sys.path.append(os.path.abspath(".."))  
from Resources.properly_format_data import GetData

In [47]:
pd.set_option('display.max_columns', None)

In [48]:
non_pa_list = ['PA', 'AB']
pa_list = ['R', 'H', '1B', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'GDP', 'HBP', 'SH', 'SF', 'IBB']
labels = ['IDfg', 'Season', 'Name', 'Team', 'Age']
combined_list = pa_list

In [49]:
data_set = GetData(2025, 2025, non_pa_list, pa_list, labels)

In [50]:
formatted_df = data_set.format_data_for_models(add_2026=True)

In [51]:
formatted_df = formatted_df.fillna(0)

In [52]:
marcel_df = formatted_df[labels+['PA']].copy()

for stat in pa_list:
    player_weighted_stat = ((5*(formatted_df[f'1Prev_{stat}'])) + (4*(formatted_df[f'2Prev_{stat}'])) + (3*(formatted_df[f'3Prev_{stat}'])))/12

    player_weighted_pa = ((formatted_df['1Prev_PA']*5) + (formatted_df['2Prev_PA']*4) + (formatted_df['3Prev_PA']*3))
    league_rate = (
        (formatted_df[f'Prev_1yr_League_Totals_{stat}'] / formatted_df['Prev_1yr_League_Totals_PA']) * formatted_df['1Prev_PA'] * 5 +
        (formatted_df[f'Prev_2yr_League_Totals_{stat}'] / formatted_df['Prev_2yr_League_Totals_PA']) * formatted_df['2Prev_PA'] * 4 +
        (formatted_df[f'Prev_3yr_League_Totals_{stat}'] / formatted_df['Prev_3yr_League_Totals_PA']) * formatted_df['3Prev_PA'] * 3
    ) / player_weighted_pa

    regressed_rate = (player_weighted_stat + (league_rate*100))/((player_weighted_pa/12)+100)

    pa_proj = (0.5*formatted_df['1Prev_PA'])+(0.1*formatted_df['2Prev_PA'])+200
    age_adj = np.where(
        formatted_df['Age'] > 29,
        1 / (1 + 0.003 * (formatted_df['Age'] - 29)),
        np.where(
            formatted_df['Age'] < 29,
            1 + 0.006 * (29 - formatted_df['Age']),
            1
        )
    )
    if stat ==  'SO' or stat == 'CS':
        age_adj = 1/age_adj

    age_adj_rate = regressed_rate*age_adj
    weighted_league_rate = (
        (5 * (formatted_df[f'Prev_1yr_League_Totals_{stat}'] / formatted_df['Prev_1yr_League_Totals_PA'])) +
        (4 * (formatted_df[f'Prev_2yr_League_Totals_{stat}'] / formatted_df['Prev_2yr_League_Totals_PA'])) +
        (3 * (formatted_df[f'Prev_3yr_League_Totals_{stat}'] / formatted_df['Prev_3yr_League_Totals_PA']))
    ) / (12)

    weighted_value = np.sum(age_adj_rate * pa_proj) / np.sum(pa_proj)
    formatted_df['weighted_value'] = weighted_value

    rebaseline = weighted_league_rate/weighted_value


    
    final_rate = age_adj_rate*rebaseline
    
    
    
    proj_stat = pa_proj*age_adj_rate
    marcel_df[stat] = age_adj_rate
    
    

In [53]:
marcel_df = marcel_df[marcel_df['Season'] == 2025]

In [54]:
for stat in pa_list:
    marcel_df[stat] = marcel_df[stat] * marcel_df['PA']

In [55]:
marcel_df['AB'] = marcel_df['PA'] - (marcel_df['BB'] + marcel_df['HBP'] + marcel_df['SF'] + marcel_df['SH'])

In [56]:
marcel_df['wOBA'] = ((.691*marcel_df['BB']) + (.722*marcel_df['HBP']) + (.882*marcel_df['1B']) + (1.252*marcel_df['2B']) + (1.584*marcel_df['3B']) + (2.037*marcel_df['HR']))/(marcel_df['AB'] + marcel_df['BB'] - marcel_df['IBB'] + marcel_df['SF'] + marcel_df['HBP'])

In [58]:
marcel_df.head()

,IDfg,Season,Name,Team,Age,PA,R,H,1B,2B,3B,HR,RBI,SB,CS,BB,SO,GDP,HBP,SH,SF,IBB,AB,wOBA
12,10155,2025,Mike Trout,LAA,33.0,556.0,80.243248,124.204246,66.638648,22.740830,2.814615,32.010153,71.486973,7.053361,1.253622,60.068059,145.984073,6.472425,7.428710,0.314448,2.196160,4.619899,485.992623,0.369793
20,10200,2025,Tucker Barnhart,TEX,34.0,15.0,1.323871,2.896635,2.149366,0.522025,0.021148,0.204096,1.184773,0.138110,0.027884,1.371517,4.010194,0.282258,0.097893,0.075090,0.057815,0.015156,13.397686,0.269418
23,10231,2025,Jose Iglesias,SDP,35.0,343.0,39.529846,91.725788,67.042417,18.811714,0.716338,5.155318,33.932673,4.562871,2.157008,17.055755,53.341700,6.405941,6.373376,0.214064,1.872648,0.611044,317.484157,0.323539
28,10243,2025,Randal Grichuk,- - -,33.0,293.0,37.138630,70.822030,42.868828,16.131590,1.496427,10.325184,36.134630,1.947241,0.668598,18.436409,61.609986,6.040692,3.351755,0.131729,1.525930,0.531098,269.554177,0.330335
38,10324,2025,Marcell Ozuna,ATL,34.0,592.0,76.973579,141.981093,83.920195,25.998382,0.583373,31.479143,83.663086,2.252865,0.627011,54.072999,142.532310,14.285625,3.178692,0.193688,3.293232,1.177537,531.261389,0.357711


In [60]:
marcel_df.to_csv('./correct wOBA comparison/my_system.csv', index=False)